Training aşamasına geçmeden önce bu aşamada kullandığımız dosyaları ve bunların ne işe yaradıklarına bakalım.

# config.py: bütün pipeline'ın kontrol paneli
model boyutları, eğitim ayarları, dosya yolları, cihaz gibi ayarları tek bir dosyada tutarak tutarlılığı sağladığımız dosya. Bu dosyayı @dataclass annotation ile yazdık.

bu dosyada olan parametreler:

Model Boyutları:

- visual_dim=2048
- text_dim=768
- audio_dim=768
- d_model=128
- hidden_dim=192
- num_heads=4
- num_layers=4
- window_size=[5,15,45,0]
- dropout=0.1
- max_seq_len=10000

Eğitim Ayarları:

- device=mps
- num_epochs=100
- patience=20
- learning_rate=1e-4
- weight_decay=1e-4
- optimizer=adamw
- scheduler=cosine
- batch_size=1
- get_attn_weights=False
- wandb=False


# dataset.py

config dosyasındaki ayarlara göre videoları tek tek okuyup modele veren dosyadır. Bu dosyayı direkt olarak orjinal modelden alıp kendi modelimize koyduk.

dataset.py, PyTorch'un veri soyutlaması: "kaç örnek var" (__len__) ve "i. örneği ver" (__getitem__) sorularını cevaplar. Bizim durumumuzda bir örnek = bir maç. __getitem__(i)

i. maçı HDF5 dosyalarından okuyup tensörlere çevirir ve modele hazır hâlde döndürür. Hangi maçların bu split'e (train/val/test) ait olduğunu trt_split.json'dan okur

Yani kabaca, bir maçı okuyup beş tensöre çeviren PyTorch Dataset'i. Beş anahtarı koşulsuz okuduğu için hepsi HDF5'te var olmak zorunda.

dataset.py bize tek bir maçı veriyor, fakat eğitim tek tek maçlarla değil, batchlerle çalışır. Yani DataLoader birden fazla örneği alır ve tek bir tensöre yığar, modele topluca verir. Burada bir problemimiz var. Her videonun uzunlugu (N) aynı değil. Farklı uzunlıkları nasıl batch yapacağız ?

## padding + CollateFn

çözümü 2 parça halinde bakalım. ilk olarak padding işlemi. Batchteki en uzun videonun uzunluğunu alıp, kısaları onun uzunluğu kadar sahte satırlarla doldur. Böylece hepsi N'ye gelir ve batchleyebiliriz. Bu doldurma işlemini yapan fonksiyon CollateFn. Fakat bu padding işlemi bir problem daha yaratıyor - mask'

### mask

Batche eklediğimiz o sahte satırlar loss hesabında ve metriklerde hata verir. bu sebeple onları maskelemeliyiz. Yani modele hangi satırların gerçek hangilerinin sahte olduğunu söyleyen bir boolean dizi yaratacağız. CollateFn, padding yaparken bu mask'i de üretir ve batch'le birlikte döndürür.

Mask nasıl çalışır ?

loss hesaplanırken model çıktısını ve etiketi yalnızca gerçek satırlarda karşılaştırıyoruz. criterion(output[mask], gt[mask]). Yani output[mask] sadece gerçek saniyeleri seçiyor, padding'i dışarıda bırakıyor. Böylece model olmayan veriden ceza yemiyor, sadece gerçek içerikten öğreniyor. Aynı mask, metrik hesabında da gerçek/sahte ayrımını sağlıyor.


# solver.py

Solver bir sınıftır. bir nevi beyin gibidir. Şöyle düşünebiliriz:

Elimizde üç şey var: bir model (henüz aptal, sadece transfer edilmiş ağırlıklarla başlamış), bir veri (maçlar + doğru cevaplar), ve bir hedef (modeli bu veriyle akıllandırmak). Ama bu üçü kendi kendine bir araya gelmez. Birinin oturup şunları tekrar tekrar yapması gerekir:

- Veriden bir maç al, modele ver, tahminini iste.
- Tahmini doğru cevapla karşılaştır, ne kadar yanlış olduğunu ölç.
- Bu yanlışa göre modelin ağırlıklarını azıcık düzelt.
- Bunu binlerce kez tekrarla, model gitgide daha az yanılana kadar.
- Arada sınav yap (val), en iyi hâlini kaydet, iyileşme durunca dur.

işte tekrarlayan bu işi yürüten yönetici solver'dır.


# train.py

train yukarıdaki bütün parça methodları birbirine bağlar, tek bir dosyada çalıştırır.

